# 📐 Semana 2 · Unidad 1 — Diseño de Algoritmos: Clase Esencial

**Algoritmos y Estructuras de Datos · Universidad de Talca**

---

Notebook de acompañamiento de la cátedra. Cada sección toma **un paradigma** y lo trabaja
sobre **un problema completo**, con el código que se discute en clase.

| # | Paradigma | Problema | Tiempo |
|---|---|---|---|
| 1 | **Dividir para conquistar** | Potencia rápida xⁿ | ~20 min |
| 2 | **Codicioso** | Selección de actividades | ~20 min |
| 3 | **Programación dinámica** | Cambio de monedas mínimo | ~20 min |
| 4 | **Backtracking** | Coloración de grafos | ~20 min |
| — | Cierre | Tabla comparativa | ~5 min |

> Los ejercicios al final de cada sección son para **trabajo autónomo**; no se desarrollan
> en clase.
>
> 📌 **Ramificación y poda** no entra en esta clase. Es la extensión natural del
> backtracking —el mismo esqueleto más una función de cota— y está tratada en
> [`material_detallado/05_ramificacion_y_poda.ipynb`](material_detallado/05_ramificacion_y_poda.ipynb).

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Comprender** la idea central de cuatro estrategias de diseño trabajando un problema completo con cada una.
2. **Identificar** la señal que delata a cada paradigma en un problema nuevo.
3. **Implementar** la potencia rápida, la selección de actividades, el cambio de monedas con tabla y la coloración de grafos.
4. **Analizar** por qué la estrategia codiciosa es óptima en la selección de actividades pero no en el cambio de monedas.
5. **Resolver** los ejercicios de cierre de cada sección de forma autónoma.

In [ ]:
#!pip install --upgrade pip
#!pip install numpy
#!pip install matplotlib

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import time, random, math
from typing import List, Dict, Tuple, Optional

print('✓ Listo.')

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: *"La selección de actividades y el cambio de monedas se resuelven los dos tomando 'lo mejor de cada paso'. En uno funciona y en el otro no. ¿Qué los distingue?"* — La respuesta es la propiedad de elección codiciosa, y es el punto de la clase.

---
## Sección 1 — Divide y Vencerás
### Problema: Potencia rápida — calcular xⁿ

**Idea:** si n es par, xⁿ = (x^(n/2))² → solo necesitamos calcular la mitad.
Si n es impar, xⁿ = x · x^(n−1). En cada paso el problema se divide a la mitad → **O(log n)**.

binario(100010)=34 ; 34*2 = binario(100010) <<1 = binario(1000100) = 68
34+4= binario(100010) <<2 = binario(10001000) = 136

In [ ]:
def potencia_lenta(x: float, n: int) -> float:
    '''
    Calcula x^n multiplicando x exactamente n veces.

    Complejidad:
        Tiempo: O(n)
        Espacio: O(1)
    '''
    resultado = 1.0
    for _ in range(n): # iteramos n veces
        resultado *= x # multiplicamos x por sí mismo n veces
    return resultado


def potencia_rapida(x: float, n: int) -> float:
    '''
    Calcula x^n usando Divide y Vencerás.
    Caso base: n == 0 → 1.
    Paso D&V:  n par  → (x^(n/2))²
               n impar → x · x^(n-1)

    Complejidad:
        Tiempo: O(log n)
        Espacio: O(log n)  pila de recursión
    '''
    if n == 0:
        return 1.0 # caso base: x^0 = 1
    if n % 2 == 0: # n par
        mitad = potencia_rapida(x, n // 2) # calculamos x^(n/2) una sola vez
        return mitad * mitad          # un solo producto, no dos llamadas
    return x * potencia_rapida(x, n - 1) # una llamada recursiva menos que la versión lenta


# Verificar que dan el mismo resultado
for x, n in [(2, 10), (3, 8), (1.5, 6)]:
    lenta  = potencia_lenta(x, n)
    rapida = potencia_rapida(x, n)
    igual  = '✓' if abs(lenta - rapida) < 1e-9 else '✗'
    print(f'{x}^{n} = {rapida:.2f}  {igual}')

In [ ]:
# ── Árbol de recursión de potencia_rapida(2, 8) ─────────────────────────────
# Cada nivel divide el exponente por dos: por eso la altura es log2(n).

def arbol_potencia(x, n, profundidad=0):
    """Imprime el árbol de llamadas, una línea por llamada, con sangría por nivel."""
    sangria = "    " * profundidad
    if n == 0:
        print(f"{sangria}{x}^0 = 1                      <- caso base")
        return 1
    if n % 2 == 0:
        print(f"{sangria}{x}^{n} = ({x}^{n//2})²            [par: una llamada]")
        mitad = arbol_potencia(x, n // 2, profundidad + 1)
        return mitad * mitad
    print(f"{sangria}{x}^{n} = {x} · {x}^{n-1}          [impar: saco una x]")
    return x * arbol_potencia(x, n - 1, profundidad + 1)


print("Árbol de llamadas para 2^8:\n")
resultado = arbol_potencia(2, 8)
print(f"\nResultado: {resultado}")
print("\nProfundidad del árbol = 4 = log2(8). Cada nivel es UNA multiplicación,")
print("no dos: por eso el costo es O(log n) y no O(n).")

In [ ]:
# ── Comparación de tiempos: O(n) vs O(log n) ────────────────────────────────
N = 10**6

t0 = time.perf_counter()
potencia_lenta(1.000001, N)
t_lenta = time.perf_counter() - t0

t0 = time.perf_counter()
potencia_rapida(1.000001, N)
t_rapida = time.perf_counter() - t0

print(f'n = {N:,}')
print(f'potencia_lenta  : {t_lenta*1000:8.2f} ms   (O(n)     → {N:,} operaciones)')
print(f'potencia_rapida : {t_rapida*1000:8.4f} ms   (O(log n) → {int(math.log2(N)):,} operaciones)')
print(f'\nFactora de aceleración: {t_lenta/t_rapida:.0f}×')
print()
print('Aplicación real: RSA cifra con exponentes de miles de dígitos.')
print('Sin potencia rápida, cifrar un solo mensaje tomaría años.')

**Mensaje clave:** Dividir el tamaño del problema a la mitad en cada paso es la esencia de D&V.
O(n) → O(log n) es una diferencia de millones cuando n es grande.

### Ejercicio 1 ⭐ — Multiplicación rusa de campesinos

La *multiplicación rusa* calcula `a × b` sin usar multiplicación directa:

```
resultado = 0
mientras b > 0:
    si b es impar:  resultado += a
    a = a × 2          (desplazamiento de bits)
    b = b // 2         (división entera)
```

- **Input:** dos enteros positivos `a`, `b`
- **Output:** `a × b` (entero)
- **Pregunta:** ¿cuál es la complejidad? ¿qué paradigma usa?

In [ ]:
def multiplicacion_rusa(a: int, b: int) -> int:
    '''
    Calcula a × b sin usar el operador *.

    Complejidad:
        Tiempo: O(???)
        Espacio: O(1)
    '''
    raise NotImplementedError('Implementa multiplicacion_rusa')

In [ ]:
# ── Verificador ─────────────────────────────────────────────────────────────
casos = [(3, 5, 15), (7, 8, 56), (13, 11, 143), (1, 100, 100), (0, 50, 0)]
ok = True
for a, b, esperado in casos:
    try:
        res = multiplicacion_rusa(a, b)
    except NotImplementedError:
        print('Implementa la función primero.')
        ok = False; break
    if res != esperado:
        print(f'✗ {a} × {b}: esperado {esperado}, obtenido {res}')
        ok = False
    else:
        print(f'✓ {a} × {b} = {res}')
if ok:
    print('\n✅ Correcto. Es O(log b) — idéntica idea que potencia_rapida.')

In [ ]:
# ── Solución (descomentar para ver) ─────────────────────────────────────────
# def multiplicacion_rusa(a, b):
#     resultado = 0
#     while b > 0:
#         if b % 2 == 1:       # b es impar: contribuye a
#             resultado += a
#         a *= 2               # duplicar a
#         b //= 2              # dividir b a la mitad
#     return resultado
#
# Complejidad: O(log b) — el bucle itera log2(b) veces.
# Paradigma: Divide y Vencerás (reduce b a la mitad en cada paso).
print('Solución comentada — descomentar para ver.')

---
## Sección 2 — Greedy
### Problema: Selección de actividades

Dadas n charlas con hora de inicio y fin, selecciona el **máximo número de charlas** que caben en una sola sala.

**Decisión greedy:** siempre elegir la charla que termina más temprano (deja más hueco para las siguientes).

In [ ]:
def actividades_greedy(
        actividades: List[Tuple[int, int, str]]
) -> List[Tuple[int, int, str]]:
    '''
    Selecciona el máximo número de actividades sin solapamiento.
    Criterio greedy: ordenar por tiempo de FIN ascendente.

    Parámetros:
        actividades: lista de (inicio, fin, nombre)

    Complejidad:
        Tiempo: O(n log n)  por el ordenamiento
        Espacio: O(n)
    '''
    ordenadas = sorted(actividades, key=lambda a: a[1]) # ordenamos por fin
    seleccionadas = [] 
    fin_ultimo = -1 

    for inicio, fin, nombre in ordenadas: # iteramos por las actividades ordenadas por fin
        # Solo tomar si no solapa con la última seleccionada
        if inicio >= fin_ultimo:
            seleccionadas.append((inicio, fin, nombre)) # seleccionamos esta actividad
            fin_ultimo = fin

    return seleccionadas


CHARLAS = [
    (1, 4,  'Charla A'),
    (3, 5,  'Charla B'),
    (0, 6,  'Charla C'),
    (5, 7,  'Charla D'),
    (3, 9,  'Charla E'),
    (6, 10, 'Charla F'),
    (8, 11, 'Charla G'),
    (8, 12, 'Charla H'),
    (2, 14, 'Charla I'),
]

seleccion = actividades_greedy(CHARLAS)
print(f'Total charlas: {len(CHARLAS)}')
print(f'Seleccionadas: {len(seleccion)}')
for inicio, fin, nombre in seleccion:
    print(f'  {nombre}: [{inicio}, {fin}]')

In [ ]:
# ── Línea de tiempo de las charlas (texto) ──────────────────────────────────
seleccion = actividades_greedy(CHARLAS)
elegidas = {nombre for _, _, nombre in seleccion}

t_max = max(fin for _, fin, _ in CHARLAS)
print(f"{'charla':<12}{'inicio':>7}{'fin':>5}  " + "".join(f"{h:<2}" for h in range(t_max + 1)))
print("-" * (26 + 2 * (t_max + 1)))

for inicio, fin, nombre in sorted(CHARLAS, key=lambda a: a[1]):
    barra = "".join("██" if inicio <= h < fin else "  " for h in range(t_max + 1))
    marca = "✅" if nombre in elegidas else "  "
    print(f"{marca}{nombre:<10}{inicio:>7}{fin:>5}  {barra}")

print(f"\nSeleccionadas ({len(seleccion)}): {[n for _, _, n in seleccion]}")
print("\n👉 El criterio codicioso es «la que termina antes». Cada vez que elijo la de fin")
print("   más temprano, dejo la mayor cantidad posible de tiempo libre para las demás.")

**Mensaje clave:** Greedy funciona cuando existe un criterio de selección local que *garantiza* el óptimo global. Aquí, elegir siempre la charla que termina antes deja el mayor hueco posible para las siguientes — esto se puede probar matemáticamente.

### Ejercicio 2 ⭐ — Cambio de monedas con Greedy

Implementa `monedas_greedy(monto, denominaciones)` que devuelva la lista de monedas usadas con la estrategia greedy (siempre usar la moneda más grande que quepa).

- **Input:** `monto: int`, `denominaciones: List[int]` (ordenadas descendente)
- **Output:** `List[int]` con las monedas usadas
- Prueba con `[25, 10, 5, 1]` para monto 41 → debe dar `[25, 10, 5, 1]`
- Prueba con `[4, 3, 1]` para monto 6 → da `[4, 1, 1]` (3 monedas), pero el óptimo es `[3, 3]` (2 monedas)
- **Pregunta:** ¿por qué falla en el segundo caso? Anota tu respuesta como comentario.

In [ ]:
def monedas_greedy(monto: int, denominaciones: List[int]) -> List[int]:
    '''
    Devuelve la lista de monedas usando la estrategia greedy.

    Complejidad:
        Tiempo: O(monto × |denominaciones|)
        Espacio: O(monto)
    '''
    raise NotImplementedError('Implementa monedas_greedy')

In [ ]:
# ── Verificador ─────────────────────────────────────────────────────────────
casos = [
    (41,  [25, 10, 5, 1], 41),
    (30,  [25, 10, 5, 1], 30),
    (6,   [4,  3, 1],      6),   # greedy falla → da >2 monedas pero la suma debe ser 6
    (100, [50, 25, 10, 5, 1], 100),
]
ok = True
for monto, dens, esperado_suma in casos:
    try:
        res = monedas_greedy(monto, dens)
    except NotImplementedError:
        print('Implementa la función primero.'); ok = False; break
    if sum(res) != esperado_suma:
        print(f'✗ monto={monto}: la suma de monedas es {sum(res)}, esperado {esperado_suma}')
        ok = False
    else:
        print(f'✓ monto={monto}, dens={dens} → {res} ({len(res)} monedas)')
if ok:
    print('\n✅ La suma es correcta en todos los casos.')
    print('Nota: para [4,3,1] con monto=6 greedy da 3 monedas; el óptimo son 2 (→ DP en Sección 3).')

In [ ]:
# ── Solución (descomentar para ver) ─────────────────────────────────────────
# def monedas_greedy(monto, denominaciones):
#     dens = sorted(denominaciones, reverse=True)
#     usadas = []
#     for moneda in dens:
#         while monto >= moneda:
#             usadas.append(moneda)
#             monto -= moneda
#     return usadas
#
# Falla con [4,3,1] y monto=6 porque la moneda de 4 parece la mejor
# opción local, pero no permite llegar al óptimo global (3+3=6).
# Greedy no tiene forma de "deshacer" la elección de 4.
print('Solución comentada — descomentar para ver.')

---
## Sección 3 — Programación Dinámica
### Problema: Cambio de monedas mínimo

El mismo problema donde Greedy falla: con denominaciones `[1, 3, 4]` y monto 6, Greedy da 3 monedas (4+1+1), pero el óptimo son 2 (3+3).

**Idea DP:** guardar en una tabla `dp[m]` = mínimo de monedas para el monto `m`. Llenar de 0 a `monto`.

In [ ]:
def cambio_dp(
        monto: int,
        denominaciones: List[int],
        verbose: bool = False
) -> Tuple[int, List[int]]:
    '''
    Cambio mínimo de monedas usando Programación Dinámica.
    dp[m] = mínimo de monedas para el monto m.
    Reconstrucción: seguir qué moneda se usó en cada posición.

    Complejidad:
        Tiempo: O(monto × |denominaciones|)
        Espacio: O(monto)
    '''
    INF = float('inf') # representa un monto imposible de alcanzar
    dp   = [INF] * (monto + 1) # dp[m] = mínimo de monedas para monto m
    usada = [-1]  * (monto + 1)   # qué moneda se usó para llegar a m
    dp[0] = 0 # caso base: 0 monedas para monto 0

    for m in range(1, monto + 1): # iteramos por cada monto desde 1 hasta el monto objetivo
        for moneda in denominaciones:
            if moneda <= m and dp[m - moneda] + 1 < dp[m]: # si podemos usar esta moneda y mejora el resultado
                dp[m]    = dp[m - moneda] + 1
                usada[m] = moneda
        if verbose:
            val = dp[m] if dp[m] < INF else '∞'
            print(f'  dp[{m:2d}] = {val}')

    # Reconstruir cuáles monedas se usaron
    monedas_usadas = []
    m = monto
    while m > 0 and usada[m] != -1:
        monedas_usadas.append(usada[m])
        m -= usada[m]

    return dp[monto], monedas_usadas


# Demostrar el caso donde Greedy fallaba
DENS = [1, 3, 4]
MONTO = 6

n_monedas, monedas = cambio_dp(MONTO, DENS, verbose=True)
print(f'\nResultado: {n_monedas} monedas → {monedas}')
print(f'Greedy daba: 3 monedas → [4, 1, 1]')
print(f'DP da:       {n_monedas} monedas → {sorted(monedas, reverse=True)}  ✓')

In [ ]:
# ── La tabla de programación dinámica, en texto ─────────────────────────────
MONTO = 11
DENS  = [1, 3, 4]

n_monedas, combinacion = cambio_dp(MONTO, DENS)

# Reconstruimos la tabla completa para mostrarla
INF = float('inf')
dp = [0] + [INF] * MONTO
for m in range(1, MONTO + 1):
    for d in DENS:
        if d <= m and dp[m - d] + 1 < dp[m]:
            dp[m] = dp[m - d] + 1

print(f"Monedas disponibles: {DENS}\n")
print("monto " + "".join(f"{m:>5}" for m in range(MONTO + 1)))
print("dp[m] " + "".join(f"{(dp[m] if dp[m] < INF else '∞'):>5}" for m in range(MONTO + 1)))
print("\nCada celda se calcula mirando SOLO celdas anteriores:")
for m in range(1, min(MONTO, 6) + 1):
    opciones = [f"dp[{m-d}]+1={dp[m-d]+1:.0f}" for d in DENS if d <= m and dp[m-d] < INF]
    print(f"  dp[{m:>2}] = min({', '.join(opciones)}) = {dp[m]:.0f}")

print(f"\nPara {MONTO}: {n_monedas} monedas -> {combinacion}")
print("\n👉 La tabla es la memoria del algoritmo. Sin ella, dp[2] se recalcularía")
print("   una y otra vez para cada monto que lo necesita.")

**Mensaje clave:** La tabla `dp[]` es la *memoria* del algoritmo. En lugar de recalcular el mínimo para cada submonto, lo guardamos y lo reutilizamos. El subproblema "¿cuántas monedas necesito para el monto m?" se solapa para distintas monedas → DP es la herramienta correcta.

### Ejercicio 3 ⭐⭐ — Escaleras

¿De cuántas formas distintas se puede subir una escalera de `n` peldaños si en cada paso se puede subir **1 o 2 peldaños**?

- Ejemplos: `n=1` → 1 forma, `n=2` → 2 formas, `n=4` → 5 formas
- **Input:** `n: int`
- **Output:** número de formas (`int`)
- **Pista:** construye la tabla `dp[0..n]` donde `dp[i]` = formas de llegar al peldaño `i`. ¿Qué patrón reconoces?

In [ ]:
def escaleras_dp(n: int) -> int:
    '''
    Cuenta formas de subir n peldaños tomando 1 o 2 a la vez.

    Complejidad:
        Tiempo: O(n)
        Espacio: O(n)  (o O(1) con la versión optimizada)
    '''
    raise NotImplementedError('Implementa escaleras_dp')

In [ ]:
# ── Verificador ─────────────────────────────────────────────────────────────
esperados = [1, 1, 2, 3, 5, 8, 13, 21, 34, 55]
ok = True
for i, esp in enumerate(esperados):
    try:
        res = escaleras_dp(i)
    except NotImplementedError:
        print('Implementa la función primero.'); ok = False; break
    if res != esp:
        print(f'✗ n={i}: esperado {esp}, obtenido {res}')
        ok = False
    else:
        print(f'✓ n={i}: {res} formas')
if ok:
    print('\n✅ ¡Correcto! ¿Reconoces la secuencia?')
    print('Son los números de Fibonacci: dp[n] = dp[n-1] + dp[n-2].')

In [ ]:
# ── Solución (descomentar para ver) ─────────────────────────────────────────
# def escaleras_dp(n):
#     if n <= 1:
#         return 1
#     dp = [0] * (n + 1)
#     dp[0] = 1   # 1 forma de estar en el peldaño 0 (sin moverse)
#     dp[1] = 1   # 1 forma de llegar al peldaño 1 (un paso de 1)
#     for i in range(2, n + 1):
#         dp[i] = dp[i-1] + dp[i-2]   # llegar desde i-1 (paso 1) o desde i-2 (paso 2)
#     return dp[n]
#
# La secuencia es Fibonacci. dp[n] = F(n+1).
# Versión O(1) espacio: solo guardar los dos últimos valores.
print('Solución comentada — descomentar para ver.')

---
## Sección 4 — Backtracking
### Problema: Coloración de grafos

Dado un grafo, asignar un color a cada nodo de modo que **ningún par de nodos adyacentes comparta color**, usando a lo más `k` colores.

Es un problema NP-completo: no existe algoritmo polinomial conocido para el caso general. Backtracking lo resuelve exactamente, pero en tiempo exponencial.

In [ ]:
def colorear_grafo(
        grafo: Dict[int, List[int]],
        k: int
) -> Optional[Dict[int, int]]:
    '''
    Colorea el grafo con a lo más k colores (backtracking).
    Retorna el diccionario nodo→color, o None si es imposible.

    Poda: si un color entra en conflicto con un vecino, se descarta.

    Complejidad:
        Tiempo: O(k^n)  peor caso (n = nodos)
        Espacio: O(n)   pila de recursión + asignación
    '''
    n = len(grafo)
    asignacion: Dict[int, int] = {}
    explorados = [0]
    podados    = [0]

    def bt(nodo: int) -> bool:
        if nodo == n:
            return True    # todos los nodos coloreados
        for color in range(k):
            explorados[0] += 1
            # Comprobar si color es válido para este nodo
            conflicto = any(
                asignacion.get(vecino) == color
                for vecino in grafo[nodo]
            )
            if conflicto:
                podados[0] += 1
                continue
            asignacion[nodo] = color
            if bt(nodo + 1):
                return True
            del asignacion[nodo]   # retroceder (backtrack)
        return False

    if bt(0): # nodo 0 es el primer nodo a colorear
        print(f'k={k}: solución encontrada. '
              f'Explorados={explorados[0]}, Podados={podados[0]}')
        return asignacion
    print(f'k={k}: imposible colorear. '
          f'Explorados={explorados[0]}, Podados={podados[0]}')
    return None


# Grafo pentágono (C5): necesita 3 colores
GRAFO_C5 = {0: [1, 4], 1: [0, 2], 2: [1, 3], 3: [2, 4], 4: [3, 0]}

for k_prueba in [4, 3,2,1]:
    resultado = colorear_grafo(GRAFO_C5, k_prueba)

In [ ]:
# ── Resultado de la coloración, en texto ────────────────────────────────────
NOMBRES_COLORES = ['Azul', 'Naranja', 'Verde', 'Morado', 'Rojo']

for k in (2, 3):
    asig = colorear_grafo(GRAFO_C5, k)
    print(f"=== Pentágono (ciclo de 5 nodos) con {k} colores ===")
    if asig is None:
        print("  ❌ imposible: no existe coloración válida\n")
        continue
    for nodo in sorted(asig):
        vecinos = GRAFO_C5[nodo]
        print(f"  nodo {nodo} -> {NOMBRES_COLORES[asig[nodo]]:<8} "
              f"(vecinos {vecinos} con colores "
              f"{[NOMBRES_COLORES[asig[v]] for v in vecinos]})")
    conflictos = sum(1 for u in GRAFO_C5 for v in GRAFO_C5[u] if asig[u] == asig[v])
    print(f"  ✅ coloración válida, {conflictos} conflictos\n")

print("👉 Un ciclo impar necesita 3 colores: con 2 el backtracking agota todas las")
print("   ramas y retorna None. Ese «agotar y retroceder» es el paradigma en acción.")

**Mensaje clave:** Backtracking = fuerza bruta con poda anticipada. Al detectar conflictos temprano se evita explorar ramas inútiles. La poda es la diferencia entre "explorar todo" y "explorar lo necesario". Para grafos grandes el problema sigue siendo exponencial, pero en práctica se recorta mucho el árbol.

### Ejercicio 4 ⭐⭐ — Coloración con conteo

Modifica la función `colorear_grafo` para que en lugar de retornar la primera solución, **cuente todas las coloraciones válidas distintas**.

- **Input:** `grafo: Dict`, `k: int`
- **Output:** `int` (número de coloraciones válidas)
- Para el pentágono con k=3 el resultado es 30.
- **Pista:** en lugar de retornar `True` al llegar al nodo n, incrementa un contador y continúa explorando.

In [ ]:
def contar_coloraciones(grafo: Dict[int, List[int]], k: int) -> int:
    '''
    Cuenta el número de coloraciones válidas del grafo con k colores.

    Complejidad:
        Tiempo: O(k^n)  peor caso
        Espacio: O(n)
    '''
    raise NotImplementedError('Implementa contar_coloraciones')

In [ ]:
# ── Verificador ─────────────────────────────────────────────────────────────
# Grafo completo K3 (triángulo): con k=3 colores hay 3!=6 coloraciones válidas
K3 = {0: [1, 2], 1: [0, 2], 2: [0, 1]}
casos = [(K3, 3, 6), (GRAFO_C5, 3, 30), (K3, 2, 0)]
ok = True
for grafo, k, esperado in casos:
    try:
        res = contar_coloraciones(grafo, k)
    except NotImplementedError:
        print('Implementa la función primero.'); ok = False; break
    nodos = len(grafo)
    status = '✓' if res == esperado else '✗'
    print(f'{status} grafo({nodos} nodos), k={k}: {res} coloraciones (esperado {esperado})')
    if res != esperado:
        ok = False
if ok:
    print('\n✅ Correcto.')

In [ ]:
# ── Solución (descomentar para ver) ─────────────────────────────────────────
# def contar_coloraciones(grafo, k):
#     n = len(grafo)
#     asignacion = {}
#     contador = [0]
#
#     def bt(nodo):
#         if nodo == n:
#             contador[0] += 1   # solución completa: contar y seguir
#             return
#         for color in range(k):
#             conflicto = any(
#                 asignacion.get(v) == color for v in grafo[nodo]
#             )
#             if not conflicto:
#                 asignacion[nodo] = color
#                 bt(nodo + 1)
#                 del asignacion[nodo]
#
#     bt(0)
#     return contador[0]
print('Solución comentada — descomentar para ver.')

---
## Cierre — Los 5 paradigmas en una tabla

In [ ]:
# ── Tabla comparativa final ─────────────────────────────────────────────────
filas = [
    ['Dividir y conquistar', 'Potencia x^n',           'O(log n)',  'Sí',      'Subproblemas independientes'],
    ['Codicioso',            'Selección actividades',  'O(n log n)','A veces', 'Elección local demostrada segura'],
    ['Prog. dinámica',       'Cambio de monedas',      'O(n·k)',    'Sí',      'Subproblemas solapados'],
    ['Backtracking',         'Coloración de grafos',   'O(k^n)',    'Sí',      'Construir bajo restricciones'],
]
cols = ['Paradigma', 'Problema de hoy', 'Complejidad', '¿Óptimo?', 'Cuándo usarlo']
anchos = [22, 24, 12, 10, 34]

print("RESUMEN — Paradigmas de diseño, semana 2\n")
print("".join(c.ljust(w) for c, w in zip(cols, anchos)))
print("-" * sum(anchos))
for f in filas:
    print("".join(str(v).ljust(w) for v, w in zip(f, anchos)))

print("\nLas dos preguntas que deciden:")
print("  1. ¿Puedo partir el problema en subproblemas del mismo tipo?")
print("  2. Si puedo: ¿los subproblemas se repiten?")
print("     no se repiten -> dividir y conquistar")
print("     se repiten    -> ¿basta la mejor decisión local?")
print("                      sí, demostrable -> codicioso")
print("                      no              -> programación dinámica")
print("  Si no puedo partirlo y hay que construir bajo restricciones -> backtracking")

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 4 | Dividir para conquistar y recurrencias |
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 14 | Programación dinámica |
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 15 | Algoritmos voraces |
| Bhargava (Grok) — *Grokking Algorithms* | 2ª ed. | Cap. 4, 8, 9 | Divide y vencerás, voraces y programación dinámica |
| Goodrich, Tamassia & Goldwasser (GTG) — *Data Structures and Algorithms in Python* | 1ª ed. | Cap. 4 | Recursión |

### Recursos gratuitos en línea

- 🌐 [VisuAlgo](https://visualgo.net/en) — visualizaciones interactivas de estructuras y algoritmos.
- 🎬 [Algorithms, Part I — Sedgewick (Princeton)](https://www.coursera.org/learn/algorithms-part1) — el curso de referencia de este ramo.
- 📁 Notebooks de profundización del curso: [`material_detallado/`](material_detallado/) — uno por estrategia.

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** ve a [codeforces.com/problemset](https://codeforces.com/problemset),
> escribe la etiqueta en **Tags** y ajusta **Rating**.

**Escala de dificultad orientativa para este curso:**

| Rating | Nivel | Descripción |
|--------|-------|-------------|
| 800 | ⭐ | Aplicación directa — la mayoría puede resolverlo |
| 1000–1200 | ⭐⭐ | Requiere una pequeña adaptación |
| 1300+ | ⭐⭐⭐ | Combina la idea con otra — desafío |

**Problemas recomendados para este tópico:**

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [4A — Watermelon](https://codeforces.com/problemset/problem/4/A) | ⭐ 800 | Entrada, proceso y salida |
| 2 | [158B — Taxi](https://codeforces.com/problemset/problem/158/B) | ⭐⭐ 1100 | Estrategia codiciosa donde hay que justificar la elección |
| 3 | [189A — Cut Ribbon](https://codeforces.com/problemset/problem/189/A) | ⭐⭐⭐ 1300 | El codicioso falla; hay que tabular |
| 4 | [455A — Boredom](https://codeforces.com/problemset/problem/455/A) | ⭐⭐⭐ 1500 | Programación dinámica sobre conteos |

⚠️ Los dos primeros son el **mínimo esperado**. Los demás son desafío opcional.